# 02. Обучение модели рекомендаций

Теперь перейдём от анализа к модели. Для каждого клиента нужно расположить 24 продукта в таком порядке, чтобы продукты, которые он подключит в следующем месяце, оказались как можно выше.

Главная метрика — MAP@7. Дополнительно смотрим Recall@7, Precision@7, покрытие каталога и macro PR-AUC, чтобы оценка не зависела от одного числа.

Ноутбук использует всех клиентов из исходного CSV. Он предназначен для исследования и не перезаписывает готовую модель. Финальный артефакт и отчёты создаются отдельным скриптом. Результаты ячеек очищены, поэтому перед просмотром чисел ноутбук нужно выполнить заново.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    candidate_root = PROJECT_ROOT.parent
    if (candidate_root / "src").is_dir():
        PROJECT_ROOT = candidate_root
    else:
        raise RuntimeError("Запустите ноутбук из корня проекта или каталога notebooks")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from bank_recommender.constants import (
    DATE_COLUMN,
    ID_COLUMN,
    PRODUCT_COLUMNS,
    PRODUCT_DESCRIPTIONS_RU,
)
from bank_recommender.data import build_temporal_pairs, read_snapshots
from bank_recommender.features import engineer_features
from bank_recommender.metrics import evaluate_ranking
from bank_recommender.model import (
    BankProductRecommender,
    PopularityRecommender,
    ranked_recommendations,
)

SEED = 42
TOP_K = 7
VALIDATION_PERIOD = "2016-04"
random.seed(SEED)
np.random.seed(SEED)

raw_data_path = Path(os.getenv("DATA_PATH", "train_ver2.csv")).expanduser()
DATA_PATH = raw_data_path if raw_data_path.is_absolute() else PROJECT_ROOT / raw_data_path
MODEL_MAX_ITER = int(os.getenv("MODEL_MAX_ITER", "12"))
if MODEL_MAX_ITER < 1:
    raise ValueError("MODEL_MAX_ITER должен быть положительным")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Датасет не найден: {DATA_PATH}")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda value: f"{value:,.5f}")

print(f"Данные: {DATA_PATH}")
print(
    "Режим загрузки: все клиенты; "
    f"seed={SEED}; max_iter={MODEL_MAX_ITER}; holdout={VALIDATION_PERIOD}->2016-05"
)

## 1. Как определяем новую покупку

Для клиента $u$ и продукта $p$ целевая переменная рассчитывается так:

$$y_{u,p,t+1}=\max(product_{u,p,t+1}-product_{u,p,t}, 0).$$

Положительным событием является только переход `0→1`. Сохранение продукта, его отсутствие и отказ от него дают ноль. Пара создаётся только для соседних календарных месяцев.

В признаки попадают профиль клиента и его портфель из месяца $t$. Значения из следующего месяца не используются. Поэтому данные делятся по времени: модель обучается на более ранних месяцах, а апрель 2016 года используется для прогноза покупок мая. Правила заполнения пропусков, масштабирования и кодирования категорий настраиваются только по обучающей части данных.

In [ ]:
snapshots = read_snapshots(DATA_PATH)
pairs = build_temporal_pairs(snapshots)
x_train, y_train, x_valid, y_valid = pairs.split(VALIDATION_PERIOD)

valid_periods = x_valid[DATE_COLUMN].dt.to_period("M")
if not valid_periods.eq(pd.Period(VALIDATION_PERIOD, freq="M")).all():
    raise AssertionError("Holdout содержит строки вне апреля 2016 года")

split_summary = pd.DataFrame(
    {
        "train": [
            len(x_train),
            x_train[ID_COLUMN].nunique(),
            str(x_train[DATE_COLUMN].min().to_period("M")),
            str(x_train[DATE_COLUMN].max().to_period("M")),
            int(y_train[PRODUCT_COLUMNS].to_numpy().sum()),
            float((y_train[PRODUCT_COLUMNS].sum(axis=1) > 0).mean()),
        ],
        "validation (апрель→май)": [
            len(x_valid),
            x_valid[ID_COLUMN].nunique(),
            str(x_valid[DATE_COLUMN].min().to_period("M")),
            str(x_valid[DATE_COLUMN].max().to_period("M")),
            int(y_valid[PRODUCT_COLUMNS].to_numpy().sum()),
            float((y_valid[PRODUCT_COLUMNS].sum(axis=1) > 0).mean()),
        ],
    },
    index=[
        "пар",
        "клиентов",
        "минимальный feature-месяц",
        "максимальный feature-месяц",
        "событий 0→1",
        "доля клиентов с покупкой",
    ],
)
display(split_summary)

## 2. Проверяем, насколько редки покупки

Посчитаем число новых подключений для каждого продукта в обучающей и проверочной выборках. Это помогает заранее увидеть продукты, для которых примеров мало.

Если у продукта нет положительных событий в конкретном периоде, мы не удаляем его из каталога. Такой результат означает лишь то, что за выбранное время новых подключений не наблюдалось.

In [ ]:
target_prevalence = pd.DataFrame(
    {
        "product": PRODUCT_COLUMNS,
        "description": [PRODUCT_DESCRIPTIONS_RU[p] for p in PRODUCT_COLUMNS],
        "train_events": [int(y_train[p].sum()) for p in PRODUCT_COLUMNS],
        "train_rate": [float(y_train[p].mean()) for p in PRODUCT_COLUMNS],
        "validation_events": [int(y_valid[p].sum()) for p in PRODUCT_COLUMNS],
        "validation_rate": [float(y_valid[p].mean()) for p in PRODUCT_COLUMNS],
    }
).sort_values("train_events", ascending=False, ignore_index=True)
display(target_prevalence.head(10))
display(target_prevalence.tail(5))

## 3. Начинаем с простого ориентира

В качестве простого ориентира (baseline) используем общую популярность новых покупок. Он считает частоту переходов `0→1` в обучающей выборке и предлагает всем клиентам один и тот же порядок продуктов.

Этот вариант прост, но полезен для проверки: более сложная модель имеет смысл только в том случае, если она показывает лучший результат на данных за следующий месяц. Уже подключённые продукты при расчёте рекомендаций исключаются.

In [ ]:
baseline = PopularityRecommender().fit(x_train, y_train)
baseline_scores = baseline.predict_scores(x_valid)
comparison = {
    "popularity": evaluate_ranking(
        y_valid, baseline_scores, owned=x_valid, k=TOP_K
    )
}
display(pd.DataFrame(comparison).T)

## 4. Обучаем основную модель

Для каждого продукта обучается отдельный `SGDClassifier` с логистической функцией потерь. Перед классификаторами выполняется общая подготовка признаков:

- создаются календарные признаки и характеристики текущего портфеля;
- числовые пропуски заполняются медианами, после чего значения масштабируются;
- категориальные пропуски получают отдельное значение, а категории кодируются;
- неизвестные категории при применении модели не вызывают ошибку.

В ноутбуке используется небольшое число итераций, чтобы исследовательский запуск не занимал слишком много времени. Финальное обучение запускается отдельным скриптом с более полными настройками.

In [ ]:
candidate = BankProductRecommender(
    alpha=0.0001,
    max_iter=MODEL_MAX_ITER,
    prior_weight=0.0,
    random_seed=SEED,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", message="Label .*", category=UserWarning)
    candidate.fit(x_train, y_train)

raw_model_scores = candidate.predict_scores(x_valid)
if raw_model_scores.shape != (len(x_valid), len(PRODUCT_COLUMNS)):
    raise AssertionError("Модель вернула матрицу score неожиданного размера")
print(f"Матрица score: {raw_model_scores.shape}")

In [ ]:
pipeline_steps = pd.DataFrame(
    [
        {"этап": name, "класс": type(step).__name__}
        for name, step in candidate.pipeline_.steps
    ]
)
preprocessor = candidate.pipeline_.named_steps["preprocessing"]
preprocessing_steps = pd.DataFrame(
    [
        {
            "ветка": name,
            "pipeline": " → ".join(step_name for step_name, _ in transformer.steps),
            "число входных признаков": len(columns),
        }
        for name, transformer, columns in preprocessor.transformers_
        if hasattr(transformer, "steps")
    ]
)
engineered_preview = engineer_features(x_train.head(3))
display(pipeline_steps)
display(preprocessing_steps)
print(
    f"После feature engineering: {engineered_preview.shape[1]} признаков; "
    f"классификаторов/меток: {len(PRODUCT_COLUMNS)}"
)

## 5. Сравниваем несколько вариантов

Для редких продуктов оценки линейной модели могут быть нестабильными. Поэтому проверим не только исходные оценки модели, но и их смеси с популярностью:

$$score=(1-w)\,score_{SGD}+w\,score_{popularity},\quad w\in\{0,0.25,0.5,0.75\}.$$

Лучший вес выбирается по MAP@7 на переходе апрель→май. Остальные метрики помогают понять результат подробнее: Recall@7 показывает долю найденных покупок, Precision@7 — полезность первых семи рекомендаций, покрытие каталога — разнообразие выдачи, а macro PR-AUC — качество по отдельным продуктам.

In [ ]:
BLEND_WEIGHTS = [0.0, 0.25, 0.5, 0.75]
for weight in BLEND_WEIGHTS:
    blended_scores = (1.0 - weight) * raw_model_scores + weight * baseline_scores
    comparison[f"sgd_prior_{weight:.2f}"] = evaluate_ranking(
        y_valid, blended_scores, owned=x_valid, k=TOP_K
    )

metric_columns = [
    "map_at_7",
    "recall_at_7",
    "precision_at_7",
    "catalog_coverage_at_7",
    "macro_pr_auc",
    "customers_with_purchase_rate",
]
comparison_frame = (
    pd.DataFrame(comparison).T[metric_columns]
    .sort_values("map_at_7", ascending=False)
)
learned_names = [name for name in comparison if name.startswith("sgd_prior_")]
best_blend_name = max(learned_names, key=lambda name: comparison[name]["map_at_7"])
best_blend_weight = float(best_blend_name.rsplit("_", 1)[1])
best_blend_scores = (
    (1.0 - best_blend_weight) * raw_model_scores
    + best_blend_weight * baseline_scores
)

display(comparison_frame)
print(f"Лучший обучаемый вариант по MAP@7: {best_blend_name}")

In [ ]:
plot_metrics = (
    comparison_frame[["map_at_7", "recall_at_7", "precision_at_7"]]
    .reset_index(names="experiment")
    .melt(id_vars="experiment", var_name="metric", value_name="value")
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_metrics, x="experiment", y="value", hue="metric")
plt.title("Качество baseline и смесей на holdout апрель→май")
plt.xlabel("Эксперимент")
plt.ylabel("Значение метрики")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Метрика")
plt.tight_layout()
plt.show()

## 6. Проверяем рекомендации

Модель не должна предлагать продукт, которым клиент уже пользуется. Следующая ячейка проверяет это правило для всей проверочной выборки и показывает один пример результата.

Заодно проверяется отсутствие повторов и то, что целевая переменная `0→1` не пересекается с текущим портфелем клиента.

In [ ]:
recommendations = ranked_recommendations(best_blend_scores, x_valid, k=TOP_K)
owned_matrix = x_valid[PRODUCT_COLUMNS].to_numpy(dtype=bool)
target_matrix = y_valid[PRODUCT_COLUMNS].to_numpy(dtype=bool)
if np.logical_and(owned_matrix, target_matrix).any():
    raise AssertionError("Target 0→1 пересекается с текущими владениями")

violations = []
for row_number, row_recommendations in enumerate(recommendations):
    owned_products = {
        product for product in PRODUCT_COLUMNS if owned_matrix[row_number, PRODUCT_COLUMNS.index(product)]
    }
    recommended_products = [item["product"] for item in row_recommendations]
    overlap = owned_products.intersection(recommended_products)
    if overlap or len(recommended_products) != len(set(recommended_products)):
        violations.append((row_number, sorted(overlap)))

if violations:
    raise AssertionError(f"Найдены нарушения фильтрации: {violations[:3]}")

example_row = 0
example_owned = [
    PRODUCT_DESCRIPTIONS_RU[p]
    for p in PRODUCT_COLUMNS
    if bool(x_valid.iloc[example_row][p])
]
example_recommended = [
    PRODUCT_DESCRIPTIONS_RU[item["product"]]
    for item in recommendations[example_row]
]
example = pd.Series(
    {
        "ncodpers": int(x_valid.iloc[example_row][ID_COLUMN]),
        "уже есть": example_owned,
        "top-7 рекомендаций": example_recommended,
    },
)
print(f"Проверено строк holdout: {len(recommendations)}; нарушений: 0")
display(example.to_frame(name="значение"))

## 7. Смотрим сохранённые результаты

Эта часть только читает `reports/model_comparison.json` и `reports/model_metadata.json`. Она не изменяет файлы и не перезаписывает `ml_models/model.joblib`.

Сохранённые сейчас модель и отчёты относятся к прежнему запуску на 1% клиентов. Чтобы получить результаты по всем клиентам, нужно заново выполнить `bash scripts/train.sh`.

In [ ]:
REPORT_DIR = PROJECT_ROOT / "reports"
comparison_path = REPORT_DIR / "model_comparison.json"
metadata_path = REPORT_DIR / "model_metadata.json"
artifact_path = PROJECT_ROOT / "ml_models" / "model.joblib"

if comparison_path.is_file():
    saved_comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    saved_comparison_frame = pd.DataFrame(saved_comparison).T
    available_metrics = [column for column in metric_columns if column in saved_comparison_frame]
    print(f"Прочитан отчёт: {comparison_path.relative_to(PROJECT_ROOT)}")
    display(saved_comparison_frame[available_metrics].sort_values("map_at_7", ascending=False))
else:
    print("reports/model_comparison.json пока отсутствует — сначала запустите полный CLI.")

if metadata_path.is_file():
    saved_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    metadata_keys = [
        "model_version",
        "created_at_utc",
        "target_definition",
        "validation_period",
        "data_scope",
        "loading_mode",
        "random_seed",
        "selected_experiment",
        "selected_prior_weight",
        "artifact_sha256",
    ]
    compact_metadata = {
        key: saved_metadata[key] for key in metadata_keys if key in saved_metadata
    }
    print(f"Прочитана карточка: {metadata_path.relative_to(PROJECT_ROOT)}")
    display(pd.Series(compact_metadata, name="значение").to_frame())
else:
    print("reports/model_metadata.json пока отсутствует — сначала запустите полный CLI.")

print(f"Готовая модель существует: {artifact_path.is_file()} (ноутбук её не изменял)")

## 8. Как запустить финальное обучение

Ноутбук нужен для анализа и сравнения подходов. Финальная модель создаётся отдельной командой из корня проекта:

```bash
DATA_PATH=train_ver2.csv bash scripts/train.sh \
  --validation-period 2016-04 \
  --max-iter 60 \
  --seed 42 \
  --artifact ml_models/model.joblib \
  --report-dir reports
```

Если задан `MLFLOW_TRACKING_URI`, скрипт также сохранит параметры, метрики, модель и её сигнатуру в MLflow. Обучение на всех клиентах может занять заметное время и требует достаточно оперативной памяти.

Скрипт сравнит простой ориентир и несколько вариантов модели, выберет лучший вариант по MAP@7, переобучит его на всех доступных парах месяцев и сохранит модель вместе с отчётами и контрольной суммой.

## 9. Итог

- Модель и простой ориентир сравниваются на одном временном периоде: апрельские данные используются для прогноза майских покупок.
- Основной выбор делается по MAP@7, а дополнительные метрики помогают заметить слабое качество отдельных продуктов или слишком однообразные рекомендации.
- Вся подготовка признаков собрана в единый процесс и настраивается только по обучающей части данных.
- Уже подключённые продукты и повторы исключаются из итоговой выдачи.
- Для воспроизводимости используется `seed=42`.
- Готовые отчёты пока отражают прежний запуск. Актуальные результаты по всем клиентам появятся после выполнения `bash scripts/train.sh`.